In [ ]:
# !pip install scikit-image

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from skimage.transform import resize
from tqdm import tqdm

Reference : https://www.kaggle.com/code/loozin/wm-811k-wafermap

In [ ]:
path = './data/LSWMD.pkl'

In [ ]:
df = pd.read_pickle(path)

print(len(df))

In [ ]:
# 라벨(failureType) 전처리
# 원본 데이터의 failureType은 [['Center']] 처럼 리스트 안에 있거나 비어있음
# 이를 일반 문자열로 변환하고, 라벨이 없는 데이터는 제외하는 과정
df['failureNum'] = df.failureType
df['trainTestNum'] = df.trianTestLabel

mapping_type = {
    'Center': 0, 'Donut': 1, 'Edge-Loc': 2, 'Edge-Ring': 3, 'Loc': 4,
    'Near-full': 5, 'Random': 6, 'Scratch': 7, 'none': 8
}

# 라벨 정보가 있는 데이터만 추출
df = df.drop(['waferIndex', 'dieSize', 'lotName'], axis=1) # 불필요한 컬럼 제거
df['failureType'] = df['failureType'].apply(lambda x: x[0][0] if len(x) > 0 else 'none')

# 불량 유형별 개수 확인
print("\n<Class Distribution>")
print(df['failureType'].value_counts())

In [ ]:
def plot_wafer_maps(df, n_samples=3):
    """
    각 불량 유형별로 n_samples개씩 샘플링하여 시각화하는 함수
    """
    failure_types = ['Center', 'Donut', 'Edge-Loc', 'Edge-Ring', 'Loc', 'Random', 'Scratch', 'Near-full']
    
    fig, ax = plt.subplots(len(failure_types), n_samples, figsize=(n_samples*3, len(failure_types)*3))
    
    for i, f_type in enumerate(failure_types):
        # 해당 불량 유형 데이터 필터링
        type_df = df[df['failureType'] == f_type]
        
        # 랜덤 샘플링
        if len(type_df) >= n_samples:
            samples = type_df.sample(n_samples)['waferMap'].values
        else:
            samples = type_df['waferMap'].values
            
        for j in range(n_samples):
            if j < len(samples):
                # 웨이퍼 맵 이미지 출력
                # cmap='inferno': 0(보라/검정-배경), 1(주황-정상), 2(노랑-불량) 등으로 표현됨
                # 직관적인 색상을 위해 사용자 정의 cmap을 써도 좋음
                ax[i, j].imshow(samples[j], cmap='inferno') 
                ax[i, j].set_title(f"{f_type}", fontsize=10)
                ax[i, j].axis('off')
            else:
                ax[i, j].axis('off')
                
    plt.tight_layout()
    plt.show()

In [ ]:
plot_wafer_maps(df, n_samples=4)

In [ ]:
# CNN 입력 처리 위한 전처리 (Resize) 
# 웨이퍼크기가 제각각이라 고정된 크기 (64*64)로 맞춤 

def preprocess_wafer_map(wafer_map, target_size=(64, 64)):
    """
    CNN 입력용으로 이미지 크기를 통일하는 함수
    OpenCV의 resize 등을 사용해도 무방
    """
    # 웨이퍼 맵을 target_size로 변환 (Nearest Neighbor 보간법 권장: 0,1,2 값을 유지하기 위해)
    # 다만, skimage resize는 기본적으로 float로 반환하므로 주의 필요
    resized_map = resize(wafer_map, target_size, anti_aliasing=False, preserve_range=True, order=0)
    
    # 0, 1, 2 정수값 유지
    return resized_map.astype(int)

In [ ]:
# 테스트: 첫 번째 데이터 변환 확인
sample_wafer = df.iloc[0]['waferMap']
resized_wafer = preprocess_wafer_map(sample_wafer)

print(f"\nOriginal shape: {sample_wafer.shape}")
print(f"Resized shape: {resized_wafer.shape}")

In [ ]:
plt.imshow(resized_wafer)

# Baseline

In [ ]:
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
tqdm.pandas()

# Diet
# 불량 데이터(Failures)는 모두 유지
# 정상 데이터(None)는 5,000개만 샘플링 
print(f"원본 데이터 크기: {len(df)}")

df_failure = df[df['failureType'] != 'none'].copy()
df_none = df[df['failureType'] == 'none'].sample(n=5000, random_state=42).copy()

df_reduced = pd.concat([df_failure, df_none])
df_reduced = df_reduced.reset_index(drop=True)

print(f"{len(df_reduced)}장으로 축소 (불량: {len(df_failure)}, 정상: 5000)")


# Preprocessing 
def preprocess_wafer_map(wafer_map, target_size=(64, 64)):
    """
    웨이퍼 맵 이미지를 target_size로 리사이징.
    - order=0: Nearest Neighbor 보간법을 사용하여 0, 1, 2 정수 라벨이 1.5 같은 실수로 변하지 않도록 함
    - anti_aliasing=False: 경계면이 흐려지는 것을 방지
    """
    # 리사이징 수행
    resized = resize(wafer_map, target_size, 
                     order=0, 
                     preserve_range=True, 
                     anti_aliasing=False)
    
    # float형을 다시 int형으로 변환 (0, 1, 2 유지)
    return resized.astype(int)


print("\n리사이징 및 변환 시작 ...")
resized_list = df_reduced['waferMap'].progress_apply(lambda x: preprocess_wafer_map(x)).tolist()

# Numpy Array 변환 시 'float32' 사용 
# 기본 float64 대신 float32를 쓰면 메모리를 절반만 사용
X = np.array(resized_list, dtype=np.float32)

# CNN 입력 차원 추가 (N, H, W) -> (N, 1, H, W)
# PyTorch: (Batch, Channel, Height, Width)
X = X[:, np.newaxis, :, :]

print("\n=== 최종 결과 확인 ===")
print(f"최종 X 데이터 Shape: {X.shape}") 
print(f"메모리 사용량: {X.nbytes / 1024 / 1024:.2f} MB")

In [ ]:
# Y 데이터 (Class Index 사용)
mapping_type = {
    'Center': 0, 'Donut': 1, 'Edge-Loc': 2, 'Edge-Ring': 3, 'Loc': 4,
    'Near-full': 5, 'Random': 6, 'Scratch': 7, 'none': 8
}
y = df_reduced['failureType'].replace(mapping_type).values.astype(int)

print(f"y shape: {y.shape}")

In [ ]:
# data set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Custom Dataset 클래스 정의
class WaferDataset(Dataset):
    def __init__(self, images, labels):
        self.images = torch.FloatTensor(images) # Float형 텐서 변환
        self.labels = torch.LongTensor(labels)  # 정수형(Long) 텐서 변환
        
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

In [ ]:
# DataLoader 생성 (배치 단위 로딩)
train_dataset = WaferDataset(X_train, y_train)
test_dataset = WaferDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self):
        super(BaselineCNN, self).__init__()
        
        # 과제 포인트 1: Conv 레이어 깊이 및 필터 수 조절
        # 입력: (Batch, 1, 64, 64)
        self.layer1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=0), # 64 -> 62
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) # 62 -> 31
        )
        
        self.layer2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=0), # 31 -> 29
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) # 29 -> 14
        )
        
        # 과제 포인트 2: Fully Connected Layer 구조 변경 (Dropout 추가 등)
        # Flatten 후 크기 계산: 64(채널) * 14(가로) * 14(세로) = 12544
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 14 * 14, 64), 
            nn.ReLU(),
            nn.Linear(64, 9) # 9개 클래스
        )
        # 주의: PyTorch CrossEntropyLoss는 내부에 Softmax를 포함하므로, 
        # 모델 끝에 Softmax를 붙이지 않습니다.

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = self.fc(out)
        return out

model = BaselineCNN().to(device)
print(model)

In [ ]:
# Training 
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001) 

def train_model(model, train_loader, criterion, optimizer, epochs=10):
    train_losses = []
    
    for epoch in range(epochs):
        model.train() 
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100 * correct / total
        train_losses.append(epoch_loss)
        
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%")
        
    return train_losses

# 학습 실행
print("Starting Training...")
history = train_model(model, train_loader, criterion, optimizer, epochs=10)

In [ ]:
# 테스트 데이터 평가 함수
def evaluate_model(model, test_loader):
    model.eval() 
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    acc = 100 * correct / total
    return acc

# 최종 성능 확인
test_acc = evaluate_model(model, test_loader)
print(f"\nFinal Test Accuracy: {test_acc:.2f}%")

# 학습 Loss 시각화
plt.plot(history, label='Train Loss')
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

---
End of Documents